# 3 — Isolated Stage 3 smoke gate
Runs a complete 12-episode ID/OOD pipeline on throwaway seed 999. Outputs go only to `~/stage3_smoke`; analysis seeds 14–21 remain untouched.

In [ ]:
import os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; OUT=Path.home()/"stage3_smoke"; P=Path.home()/"LIBERO-plus"; GPU=(Path.home()/"stage3_gpu.txt").read_text().strip()
IDPY=Path.home()/"venv-stage1-id/bin/python"; OODPY=Path.home()/"venv-stage1-ood/bin/python"; MAN=OUT/"stage3_smoke_manifest.csv"; OUT.mkdir(exist_ok=True)
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
subprocess.run([str(IDPY),"-m","async_vla_benchmark.scripts.make_stage3_smoke_manifest","--output",str(MAN),"--git-sha",bench,"--lerobot-git-sha","2aba372b4e217cc47db28e0f836859b20d1456c9","--libero-plus-git-sha",plus,"--model-revision","8e174154ef5f6c60a8da12ae99c303d8963138c1"],cwd=R,check=True)
base={"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1"}
for scene,py in (("id",IDPY),("ood",OODPY)):
    env=os.environ.copy(); env.update(base)
    if scene=="ood": env["PYTHONPATH"]=str(P)
    subprocess.run([str(py),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage3.yaml"),"--manifest",str(MAN),"--scene",scene,"--expected-rows","12"],cwd=R,env=env,check=True)


In [ ]:
for scene,py in (("id",IDPY),("ood",OODPY)):
    env=os.environ.copy(); env.update(base)
    if scene=="ood": env["PYTHONPATH"]=str(P)
    log=OUT/f"stage3_smoke_{scene}.log"; cmd=[str(py),"-u","-m","async_vla_benchmark.scripts.run_stage3","--config",str(R/"async_vla_benchmark/configs/stage3.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene",scene,"--resume","--verbose"]
    with open(log,"ab") as fh: subprocess.run(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,check=True)
    print(scene,"\n".join(log.read_text().splitlines()[-8:]))
subprocess.run([str(OODPY),"-m","async_vla_benchmark.scripts.validate_stage3_smoke","--manifest",str(MAN),"--output-dir",str(OUT)],cwd=R,env={**os.environ,**base,"PYTHONPATH":str(P)},check=True)
print("STOP HERE. Paste both log tails and the PASS line before notebook 04.")
